# Illinois walkthrough — handle, State AGR, online vs retail

Official source: [IGB sports reports](https://igb.illinois.gov/sports-wagering/sports-reports.html).

Two different official CSVs:

1. **All Wagering Activity → Sport Detail** — handle by licensee and channel
2. **Completed Events → Tax Summary** — **State AGR** and **State Tax**

State AGR is adjusted revenue. We store it in `adjusted_revenue` and do **not** copy it into
`gross_revenue`. Retail rows are used to reconcile official `Total` lines, then excluded from
the primary `online_sports_betting` dataset.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.common import project_root
from variant_gaming.states.illinois import (
    join_handle_and_revenue,
    parse_sport_detail_handle,
    parse_tax_summary,
)
from variant_gaming.storage import connect_readonly

ROOT = project_root()
database_file = "data/staging/gaming_nationwide.sqlite"  # explicitly select another snapshot if needed
database_path = ROOT / database_file
print("Read-only database:", database_path)
handle_text = (ROOT / "tests/fixtures/IL/sample_sport_detail.csv").read_text(encoding="utf-8")
tax_text = (ROOT / "tests/fixtures/IL/sample_tax_summary.csv").read_text(encoding="utf-8")
print(handle_text.splitlines()[0])
print(tax_text.splitlines()[0])

## Clean handle, then tax, then join

In [ ]:
handle_df, handle_check = parse_sport_detail_handle(handle_text)
print("handle max abs reconciling difference", handle_check["difference"].abs().max())
handle_df.head()

In [ ]:
tax_df = parse_tax_summary(tax_text)
print("negative AGR rows", int((tax_df["adjusted_revenue"] < 0).sum()))
tax_df.head()

In [ ]:
joined = join_handle_and_revenue(handle_df, tax_df)
online = joined[joined["channel"] == "online"]
print("online operators", len(online), "retail operators", int((joined["channel"]=="retail").sum()))
online.sort_values("operator").head()

Retail exists in the official file. It is **not** labeled `online_sports_betting` in the
database. The collector upserts online operator rows only.

SQLite uses a unique key of state, vertical, channel, operator, row type, period, and source hash.
Re-runs update the same key; they do not wipe New York.

`database_file` in the setup cell defaults to staging. The inspection below opens it read-only; parsing the retained CSV examples needs no database.


In [ ]:
il = pd.DataFrame()
if database_path.exists():
    conn = connect_readonly(database_path)
    try:
        il = pd.read_sql_query("SELECT channel, row_type, COUNT(*) AS n FROM gaming_results WHERE state_code='IL' GROUP BY 1, 2", conn)
    finally:
        conn.close()
else:
    print("Selected database does not exist; inspection skipped. No database was created.")
il
